# core

> Claude code api backend for fastllm

FastLLM's `claude_code` provider is a thin adapter over `fastclaude`. `claude_mk_payload` hands the canonical history and tool schemas to `astream`, and `claude_acollect_stream` normalizes the run's events to FastLLM deltas. Every request is stateless: a history ending in tool results continues by fastclaude's deferral, so FastLLM's ordinary client-owned tool loop replays canonical history and no response id is ever issued.

In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.utils import *
from fastllm.types import *
from aidialog.msg_parts import ToolUse
from fastllm.anthropic import norm_sse_event, norm_tool_calls, norm_parts, norm_finish, norm_usage, finalize_usage, delta_index_fn, collate_raw, cost
from fastllm.streaming import mk_acollect_stream, Delta
from fasttransport.errors import APIError
from fastclaude.core import astream, unqual, SERVER_TOOLS

In [ ]:
from fastllm.chat import AsyncChat, lite_mk_func, mk_msgs, mk_tool_res_msg
from fastcore.test import *

`claude_mk_payload` maps a FastLLM request onto `astream`'s arguments. Both Responses and Chat Completions function schemas map to fastclaude's, and `web_search_options` enables Claude Code's own search tools. An `oauth_token` becomes the child process's `CLAUDE_CODE_OAUTH_TOKEN`, the subscription sign-in the CLI honours in place of its stored login, so a host serving many users runs each turn on that user's subscription token rather than the machine's login. The run loads no Claude Code settings unless `setting_sources` says otherwise: a serving host's own CLAUDE.md, hooks, and skills belong to the host account, not to the user whose turn this is, and a completions call is not the place for them:

In [ ]:
#| export
def claude_mk_payload(msgs, model, stream=False, **kwargs):
    "Build `astream` inputs from a FastLLM request"
    tools = [dict(name=f['name'], description=f.get('description',''), inputSchema=f.get('parameters', {}))
        for t in (kwargs.get('tools') or []) if t.get('type') == 'function' and (f := t.get('function') or t)]
    native = SERVER_TOOLS if kwargs.get('web_search_options') is not None else ()
    payload = dict(msgs=list(msgs), model=model, system=kwargs.get('system') or '', tools=tools or None, native_tools=native,
        setting_sources=kwargs.get('setting_sources', ()))
    if key := kwargs.get('oauth_token'): payload['env'] = dict(CLAUDE_CODE_OAUTH_TOKEN=key)
    return payload

In [ ]:
def simple_add(a: int, b: int) -> int:
    "Add two numbers"
    return a + b

p = claude_mk_payload(mk_msgs(['What is 2+2?']), 'claude-sonnet-5', tools=[lite_mk_func(simple_add)])
test_eq(p['native_tools'], ())
rp = claude_mk_payload([], 'm', tools=[dict(type='function', name='py', parameters={'type':'object'})])
test_eq(rp['tools'][0]['name'], 'py')
p['tools'][0]

In [ ]:
test_eq(claude_mk_payload([], 'm', web_search_options='l')['native_tools'], SERVER_TOOLS)
test_eq(claude_mk_payload([], 'm', oauth_token='sk-ant-oat01-x')['env'], dict(CLAUDE_CODE_OAUTH_TOKEN='sk-ant-oat01-x'))
test_eq(claude_mk_payload([], 'm')['setting_sources'], ())
test_eq(claude_mk_payload([], 'm', setting_sources=['project'])['setting_sources'], ['project'])
tmsg = mk_tool_res_msg([ToolUse(id='call_1', name='simple_add', arguments={})], ['4'])
claude_mk_payload([tmsg], 'claude-sonnet-5')['msgs']

The stream adapter re-indexes partial events onto one global block sequence (`_reidx`), because thinking, text, and the tool call arrive as separate wire messages that each restart at index 0, then normalizes them to FastLLM `Delta`s with tool names unqualified.

In [ ]:
#| export
def _reidx():
    "Stateful rebase of per-message block indices onto one global sequence"
    base,mx = 0,-1
    def f(ev):
        nonlocal base,mx
        t = ev.get('type')
        if t=='message_start': base,mx = base+mx+1,-1
        elif t in ('content_block_start','content_block_delta','content_block_stop') and 'index' in ev:
            mx = max(mx, ev['index'])
            ev = {**ev, 'index': ev['index']+base}
        return ev
    return f


In [ ]:
reindex = _reidx()
indices = [reindex(dict(type='content_block_start', index=0))['index'],
    reindex(dict(type='content_block_start', index=1))['index']]
reindex(dict(type='message_start'))
indices.append(reindex(dict(type='content_block_start', index=0))['index'])
test_eq(indices, [0,1,2])
indices

Claude CLI emits one empty `partial_json` chunk for a zero-argument tool. FastLLM's collector normally recognizes a tool only when arguments finish as JSON, so `_ToolBlocks` remembers every opened tool block and which ones actually completed JSON.

In [ ]:
#| export
class _ToolBlocks:
    def __init__(self): self.open,self.complete = {},set()

`observe` updates that state after each normalized event. When an unfinished block closes, it returns one synthetic empty-arguments delta; ordinary and completed blocks return nothing extra.

In [ ]:
#| export
@patch
def observe(self:_ToolBlocks, ev, delta):
    et,idx = ev.get('type'),ev.get('index')
    if et=='content_block_start' and delta.tool_calls: self.open[idx] = delta.tool_calls[0]
    elif et=='content_block_delta' and delta.tool_calls and (nested_idx(ev, 'delta', 'partial_json') or '').endswith('}'):
        self.complete.add(idx)
    if et!='content_block_stop' or (call := self.open.pop(idx, None)) is None or idx in self.complete: return
    return Delta(tool_calls=[ToolUse(id=call.id, name=call.name, arguments={})], raw=dict(index=idx))

In [ ]:
blocks = _ToolBlocks()
empty_call = ToolUse(id='call_0', name='ping', arguments={})
test_eq(blocks.observe(dict(type='content_block_start', index=0), Delta(tool_calls=[empty_call])), None)
empty_delta = blocks.observe(dict(type='content_block_stop', index=0), Delta())
test_eq(empty_delta.tool_calls[0].arguments, {})
empty_delta

Each response is one generator pipeline: raw run events are filtered to partials, re-indexed, normalized, and collected by FastLLM's standard collector.

The calls are not marked server-executed: they are the chat's own tools, and FastLLM's loop must run them.

In [ ]:
#| export
async def _claude_deltas(run):
    "Normalize one ClaudeRun turn to FastLLM deltas"
    reindex = _reidx()
    blocks = _ToolBlocks()
    async for m in run:
        if m.get('type')!='stream_event': continue
        ev = reindex(m['event'])
        d = norm_sse_event(ev)
        for tc in (d.tool_calls or []): tc.name = unqual(tc.name)
        yield d
        if extra := blocks.observe(ev, d): yield extra

Claude CLI reports provider failures in its terminal result rather than raising them through iteration. `_check_run` translates that terminal shape to FastLLM's ordinary provider error after all streamed parts have been collected.

In [ ]:
#| export
def _check_run(run, payload):
    if not run.result or not run.result.get('is_error'): return
    raise APIError(str(run.result.get('result') or run.result.get('subtype')), provider='claude_code',
        model=payload.get('model'), status_code=run.result.get('api_error_status'), raw=run.result)

`claude_acollect_stream` is the lifecycle composition: start the run, normalize and collect its deltas, and translate a terminal provider error. The run closes itself when its event stream ends, however that happens.

In [ ]:
#| export
async def claude_acollect_stream(payload, **kwargs):
    "Adapt one `ClaudeRun` to FastLLM"
    run = astream(**payload)
    async for o in mk_acollect_stream(_claude_deltas(run), index_fn=delta_index_fn, api_name='claude_code', **kwargs): yield o
    _check_run(run, payload)

The provider registration supplies FastLLM's standard Anthropic normalization and costing functions. `collate_raw` rebuilds the assistant message in Anthropic wire form from the streamed events, which is what a replayed history hands back to `astream`. The registration declares no continuation capability, so FastLLM replays canonical history after every tool round, as it does for Codex.

In [ ]:
#| export
api_registry.register('claude_code',
    norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    finalize_usage=finalize_usage, collate_raw=collate_raw, mk_payload=claude_mk_payload, acollect_stream=claude_acollect_stream, cost=cost)

## Subscription tokens

A Claude subscription is used through a long-lived OAuth token from `claude setup-token`, which the CLI reads from `CLAUDE_CODE_OAUTH_TOKEN`. A host holding tokens for many users stores a block per user and passes the token as `oauth_token`; these three functions give that host the same shape the Codex module offers: what to store, the credential for a call, and what to show about it.

In [ ]:
#| export
def claude_auth(auth):
    "The block to store from a pasted `claude setup-token` token, or a block holding one"
    token = auth.get('token') if isinstance(auth, dict) else auth
    token = token.strip() if isinstance(token, str) else ''
    if not token.startswith('sk-ant-oat'): raise ValueError('Claude auth is the token from `claude setup-token`')
    return dict(token=token)

async def claude_token(auth, force_refresh=False):
    "The token in `auth`, and `None`: it lives for a year and there is nothing to refresh"
    return auth['token'], None

def claude_info(auth):
    "What a stored block says about itself; the token carries no plan or expiry"
    return dict(plan=None, expires=None)

In [ ]:
test_eq(claude_auth(' sk-ant-oat01-abc '), dict(token='sk-ant-oat01-abc'))
test_eq(claude_auth(dict(token='sk-ant-oat01-abc')), dict(token='sk-ant-oat01-abc'))
with expect_fail(ValueError, 'setup-token'): claude_auth('sk-ant-api03-notoauth')
await claude_token(dict(token='sk-ant-oat01-abc'))

## Live runs

Against the real CLI, one FastLLM chat drives the client-owned tool loop. The first response ends at `simple_add`; FastLLM executes it and replays the history ending in the result, which fastclaude continues by deferral in a fresh process. No response id is involved. The turn spends subscription tokens and runs only when `CLAUDE_OAUTH_TOKEN` or `CLAUDE_CODE_OAUTH_TOKEN` is set, passed as `oauth_token` the way a host would:

In [ ]:
tok = os.getenv('CLAUDE_OAUTH_TOKEN') or os.getenv('CLAUDE_CODE_OAUTH_TOKEN')
if tok:
    chat = AsyncChat('claude_code/claude-sonnet-5', tools=[simple_add], oauth_token=tok)
    rs = await chat('What is 7+3? Use the tool, then answer with only the number.', stream=True, max_steps=3)
    parts = [o async for o in rs]
    test_eq([m.role for m in chat.hist], ['user','assistant','tool','assistant'])
    test_eq(chat.hist[-1].text, '10')
    assert chat.response_id is None
[type(o).__name__ for o in parts] if tok else None

['ToolUse', 'Completion', 'ToolResult', 'Refresh', 'Text', 'Completion']